# Migración de datos de CSV a MySQL (MotoGP) con Python y SQLAlchemy

**Autores:** _Kugler Martín, Recover Diego_

En este notebook se realiza, la migración de datos desde un archivo **CSV** hacia una base de datos **MySQL**, respetando la **integridad referencial** y normalizando la información. De manera resumida, se seguirán los siguientes pasos:

1. **Configuración de la conexión** a MySQL mediante SQLAlchemy (`engine`) y preparación del entorno.
2. **Limpieza inicial de la base de datos** (vaciado de tablas con `TRUNCATE` manteniendo la estructura).
3. **Carga del CSV** en un DataFrame y **preprocesado** (estandarización de países y conversión a códigos ISO3).
4. **Migración de tablas base** (tablas sin dependencias): `countries`, `categories`, `teams`, `bikes`, `riders`, `circuits`.
5. **Creación de mapas de IDs** generados por MySQL para poder resolver claves foráneas.
6. **Migración de tablas dependientes**: inserción de `races` y finalmente `results` mediante mapeo de IDs.
7. **Comprobación final** de la migración consultando la tabla `results` desde Python.

In [ ]:
# Importamos las librerías utilizadas en la migración de datos
import pandas as pd
from sqlalchemy import create_engine, text
import country_converter as coco

---

#### **1) Creación del motor:**
Realizamos la preparación del entorno para la migración de datos:

- **Conexión a MySQL con SQLAlchemy:** se definen las credenciales (`USER`, `PASSWORD`, `HOST`, `DATABASE`) y se construye el `connection_string` para crear el **motor (`engine`)** que gestionará la conexión con la base de datos.
- **Reinicio de la migración (opción `TRUNCATE`):** si `TRUNCATE = True`, se inicia una transacción con `engine.begin()` para ejecutar comandos SQL de forma segura.
- **Desactivación temporal de llaves foráneas:** se ejecuta `SET FOREIGN_KEY_CHECKS = 0;` para evitar errores al vaciar tablas relacionadas entre sí.
- **Vaciado de tablas:** se usan sentencias `TRUNCATE TABLE` para borrar todos los datos de las tablas involucradas (`bikes`, `teams`, `categories`, `countries`, `circuits`, `races`, `results`) sin eliminar su estructura.
- **Reactivación de llaves foráneas:** se vuelve a activar la comprobación con `SET FOREIGN_KEY_CHECKS = 1;`.

In [106]:
# 1.1) Configuración de la conexión mediante el motor (engine) usando SQLAlchemy:
USER = "root"
PASSWORD = "yes"
HOST = "localhost"
DATABASE = "motogp_db"

connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}"
engine = create_engine(connection_string)

# 1.2) Eliminamos los datos de las tablas sin eliminar su estructura para iniciar
# la migración desde cero: 
TRUNCATE = True

# Usamos un bloque with para manejar la conexión:
if TRUNCATE: 
    with engine.begin() as conn:
        # Desactivar temporalmente revisión de llaves foráneas para evitar bloqueos de 
        # seguridad y permitir la limpieza de los datos sin ningún inconveniente:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        
        # Truncar (vaciar) todas las tablas involucradas:
        conn.execute(text("TRUNCATE TABLE bikes;"))
        conn.execute(text("TRUNCATE TABLE teams;"))
        conn.execute(text("TRUNCATE TABLE categories;"))
        conn.execute(text("TRUNCATE TABLE countries;"))
        conn.execute(text("TRUNCATE TABLE circuits;"))
        conn.execute(text("TRUNCATE TABLE races;"))
        conn.execute(text("TRUNCATE TABLE results;"))

        # Volver a activar la revisión de llaves foráneas:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    
    print("Conexión configurada con éxito. Base de datos vaciada y lista para una nueva migración limpia.")

else: 
    print("Conexión configurada con éxito.")

Conexión configurada con éxito. Base de datos vaciada y lista para una nueva migración limpia.


#### **2) Lectura del CSV:**

Preparamos el dataset antes de insertarlo en la base de datos:

- **Carga del fichero CSV:** se lee el archivo `moto_results.csv` en un `DataFrame` de pandas (`df`).
- **Corrección de códigos deportivos (FIM/IOC):** algunos valores de `rider_country` aparecen como abreviaturas/códigos no estándar. Se define el diccionario `fim_to_standard` para **traducir esos códigos a nombres de país** y se aplica con `replace()`.
- **Conversión automática a ISO3 con `country_converter` (coco):**
  - Se crea un convertidor `CountryConverter()`.
  - Para optimizar, se extraen primero los **países únicos** de `circuit_country` y `rider_country`.
  - Se construyen diccionarios de mapeo (`circuit_map` y `rider_map`) que convierten nombres a códigos **ISO3**.
  - Se reemplazan las columnas originales usando `map()`, dejando ambos campos (`circuit_country` y `rider_country`) normalizados en formato ISO3.

In [107]:
# 2.1) Leer el CSV:
df = pd.read_csv("../data/moto_results.csv")

# 2.2) Pre-procesar códigos deportivos (FIM/IOC).
# Algunos códigos deportivos varían del estándar y deben indicarse explícitamente a nombres:
fim_to_standard = {
    'SPA': 'Spain',
    'NED': 'Netherlands',
    'GER': 'Germany',
    'SWI': 'Switzerland',
    'RSA': 'South Africa',
    'POR': 'Portugal',
    'MAL': 'Malaysia',
    'INA': 'Indonesia',
    'RSM': 'San Marino',
    'NZE': 'New Zealand', 
    'CRO': 'Croatia', 
    'SLO': 'Slovenia', 
    'DAN': 'Denmark'
}

df['rider_country'] = df['rider_country'].replace(fim_to_standard)

# 2.3) Usar coco (country_converter) para convertir automáticamente todos los códigos de países a ISO3: 
# 2.3) Extraer solo los países únicos para convertir súper rápido
cc = coco.CountryConverter()

unique_circuits = df['circuit_country'].dropna().unique()
circuit_map = dict(zip(unique_circuits, cc.convert(names=unique_circuits.tolist(), to='ISO3')))
df['circuit_country'] = df['circuit_country'].map(circuit_map)

unique_riders = df['rider_country'].dropna().unique()
rider_map = dict(zip(unique_riders, cc.convert(names=unique_riders.tolist(), to='ISO3')))
df['rider_country'] = df['rider_country'].map(rider_map)

df.head(1)

# A continuación seguiremos las pautas de la integridad referencial para ir creando las tablas
# una a una según las dependencias de cada una de ellas.

,year,sequence,category,rider_first_name,rider_last_name,rider_number,rider_country,team_name,bike,position,points,speed,time,race_name,circuit_name,circuit_country,date
0,2021,16,Moto2,Fermín,Aldeguer,54.0,ESP,+EGO Speed Up,Boscoscuro,16,0,154.4,+37.590,Gran Premio Nolan Del Made In Italy E Dell'Emi...,Misano World Circuit Marco Simoncelli,ITA,2021-10-24


#### **3) Countries:**

Construimos la tabla de países a partir del dataset y la insertamos en MySQL:

- **Extracción de países únicos:** se toma la columna `rider_country` como referencia, se eliminan duplicados (`drop_duplicates()`) y valores nulos (`dropna()`), obteniendo un listado único de países.
- **Renombrado de columna:** la columna resultante se renombra a `id_country`, que actuará como identificador (por ejemplo, el código ISO3).
- **Creación de `country_name`:** se añade una segunda columna `country_name` inicializada con el mismo valor que `id_country`. Esto permite completar/enriquecer posteriormente con nombres completos si se desea.
- **Inserción en la base de datos:** se utiliza `to_sql()` para insertar los registros en la tabla `countries` (modo `append`, sin incluir índice).

In [108]:
# 3.1) Extraer países únicos para la tabla 'countries':
# Usamos 'rider_country' como fuente para id_country:
countries = df[['rider_country']].drop_duplicates().dropna()
countries.columns = ['id_country']

# Añadimos una columna igual que el identificador para el nombre, 
# que se podrá enriquecer mediante los nombres completos postre:
countries['country_name'] = countries.id_country

# 3.2) Insertar en la base de datos:
countries.to_sql('countries', con=engine, if_exists='append', index=False)
print("Países migrados exitosamente.")

Países migrados exitosamente.


#### **4) Categories:**

Generamos la tabla de categorías a partir del CSV y la insertamos en la base de datos:

- **Extracción de categorías únicas:** se selecciona la columna `category` del `DataFrame`, se eliminan duplicados (`drop_duplicates()`) y valores nulos (`dropna()`), obteniendo una lista única de categorías.
- **Renombrado a identificador:** la columna se renombra a `id_category`, que funcionará como identificador de la categoría.
- **Creación de `category_name`:** se añade la columna `category_name` inicializándola con el mismo valor que `id_category`, dejando preparada la tabla para enriquecer nombres/descripciones más adelante si fuese necesario.
- **Inserción en MySQL:** se utiliza `to_sql()` para añadir los registros a la tabla `categories` (`if_exists='append'`) sin incluir el índice del `DataFrame`.

In [109]:
# 4.1) Extraer categorías únicas para la tabla 'categories':
# Usamos 'category' como fuente para id_country:
categories = df[['category']].drop_duplicates().dropna()
categories.columns = ['id_category']

# Añadimos una columna igual que el identificador para el nombre, 
# que se podrá enriquecer mediante los nombres completos postre:
categories['category_name'] = categories['id_category']

# 4.2) Insertar en la base de datos:
categories.to_sql('categories', con=engine, if_exists='append', index=False)
print("Categorías migradas exitosamente.")

Categorías migradas exitosamente.


#### **5) Teams:**

Migramos la información de equipos desde el CSV:

- **Extracción de equipos únicos:** se selecciona la columna `team_name` del `DataFrame` y se eliminan duplicados (`drop_duplicates()`) y valores nulos (`dropna()`), obteniendo un listado único de equipos.
- **Inserción en la base de datos:** se inserta el resultado en la tabla `teams` usando `to_sql()` en modo `append`, sin incluir el índice.
- **Generación automática del identificador:** como no se envía la columna `id_team`, MySQL asigna automáticamente el valor de la clave primaria mediante `AUTO_INCREMENT`.

In [110]:
# 5.1) Extraer equipos únicos para la tabla 'teams':
teams = df[['team_name']].drop_duplicates().dropna()

# 5.2) Insertar en la base de datos:
teams.to_sql('teams', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_team', MySQL usará el AUTO_INCREMENT automáticamente.
print("Equipos migrados exitosamente.")

Equipos migrados exitosamente.


#### **6) Bikes:**

Migramos las motocicletas (marcas/modelos) presentes en el CSV:

- **Extracción de motocicletas únicas:** se toma la columna `bike` del `DataFrame`, se eliminan duplicados (`drop_duplicates()`) y valores nulos (`dropna()`), obteniendo un listado único de motocicletas.
- **Renombrado de columna:** se renombra `bike` a `bike_name` para ajustarse al esquema de la tabla en la base de datos.
- **Inserción en MySQL:** se insertan los registros en la tabla `bikes` mediante `to_sql()` con `if_exists='append'` y sin incluir el índice.
- **Clave primaria automática:** al no proporcionar `id_bike`, MySQL lo genera automáticamente gracias a `AUTO_INCREMENT`.

In [111]:
# 6.1) Extraer motocicletas únicas para la tabla 'bikes':
bikes = df[['bike']].drop_duplicates().dropna()
bikes.columns = ['bike_name']

# 6.2) Insertar en la base de datos:
bikes.to_sql('bikes', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_bike', MySQL usará el AUTO_INCREMENT automáticamente.
print("Motocicletas migradas exitosamente.")

Motocicletas migradas exitosamente.


#### **7) Riders:**

Preparamos e insertamos los pilotos (_riders_) en la base de datos:

- **Extracción de pilotos únicos:** se seleccionan las columnas `rider_first_name`, `rider_last_name` y `rider_country`, eliminando filas duplicadas (`drop_duplicates()`) y valores nulos (`dropna()`), para obtener un listado único de pilotos.
- **Adaptación al esquema relacional:** se renombran las columnas para que coincidan con la tabla `riders`:
  - `rider_first_name` → `first_name`
  - `rider_last_name` → `last_name`
  - `rider_country` → `id_country` (clave foránea hacia `countries`)
- **Inserción en MySQL:** se insertan los registros en la tabla `riders` mediante `to_sql()` en modo `append`, sin incluir el índice.
- **Clave primaria automática:** al no enviar `id_rider`, MySQL lo asigna automáticamente con `AUTO_INCREMENT`.

In [112]:
# 7.1) Extraer pilotos únicos para la tabla 'riders':
riders = df[['rider_first_name', 'rider_last_name', 'rider_country']].drop_duplicates().dropna()
riders = riders.rename(columns={'rider_first_name': 'first_name', 
                                'rider_last_name': 'last_name', 
                                'rider_country': 'id_country'})

# 7.2) Insertar en la base de datos:
riders.to_sql('riders', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_rider', MySQL usará el AUTO_INCREMENT automáticamente.
print("Pilotos migrados exitosamente.")

Pilotos migrados exitosamente.


#### **8) Circuits:**

Construimos la tabla de circuitos a partir del CSV y los insertamos en la base de datos:

- **Extracción de circuitos únicos:** se seleccionan las columnas `circuit_name` y `circuit_country`, eliminando duplicados (`drop_duplicates()`) y valores nulos (`dropna()`), para obtener un listado único de circuitos.
- **Renombrado para integridad referencial:** se renombra `circuit_country` a `id_country`, de forma que el país del circuito quede representado como **clave foránea** hacia la tabla `countries`.
- **Inserción en MySQL:** se insertan los registros en la tabla `circuits` mediante `to_sql()` en modo `append`, sin incluir el índice.
- **Clave primaria automática:** como no se proporciona el identificador (por ejemplo `id_circuit`), MySQL lo genera automáticamente con `AUTO_INCREMENT`.

In [ ]:
# 8.1) Extraer circuitos únicos para la tabla 'circuits':
circuits = df[['circuit_name', 'circuit_country']].drop_duplicates().dropna()
circuits = circuits.rename(columns={'circuit_country': 'id_country'})

# 8.2) Insertar en la base de datos:
circuits.to_sql('circuits', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_rider', MySQL usará el AUTO_INCREMENT automáticamente.
print("Circuitos migrados exitosamente.")

Circuitos migrados exitosamente.


Para poder insertar correctamente datos en las tablas finales `races` y `results` (que dependen de varias claves foráneas), es necesario **mapear** los valores del CSV con los **IDs reales** generados automáticamente por MySQL (`AUTO_INCREMENT`).

Por eso mismo, realizamos los siguientes pasos:

- Se consultan las tablas ya migradas (`riders`, `teams`, `bikes`, `circuits`) usando `pd.read_sql()` para obtener:
  - El **ID interno** asignado por la base de datos (`id_rider`, `id_team`, `id_bike`, `id_circuit`)
  - Los campos descriptivos correspondientes (nombre y/o apellidos, nombre del equipo, moto y circuito)
- El resultado de cada consulta se guarda en DataFrames (`riders_map`, `teams_map`, `bikes_map`, `circuits_map`) que actuarán como **tablas de correspondencia** para traducir valores del CSV a IDs válidos en la BD.

In [ ]:
# Para las dos últimas tablas de 'races' y 'results' necesitamos realizar un mapeo 
# entre los valores de los IDs generados y sus valores en el DataFrame. 

# Para ello, descargamos las tablas actuales para tener los IDs generados por MySQL:
riders_map = pd.read_sql("SELECT id_rider, first_name, last_name FROM riders", con=engine)
teams_map = pd.read_sql("SELECT id_team, team_name FROM teams", con=engine)
bikes_map = pd.read_sql("SELECT id_bike, bike_name FROM bikes", con=engine)
circuits_map = pd.read_sql("SELECT id_circuit, circuit_name FROM circuits", con=engine)

print("Mapas de IDs cargados.")

Mapas de IDs cargados.


#### **9) Races:**

Migramos las carreras a la tabla `races`, resolviendo primero las dependencias por claves foráneas:

- **Extracción de carreras únicas:** se seleccionan las columnas clave (`race_name`, `date`, `year`, `sequence`, `category`, `circuit_name`) y se eliminan duplicados y nulos para obtener un listado único de carreras.
- **Mapeo del circuito (`id_circuit`):** se realiza un `merge` con `circuits_map` usando `circuit_name` para añadir el identificador `id_circuit` asignado por MySQL.
- **Mapeo de categoría:** la columna `category` se renombra a `id_category`, ya que en este modelo el identificador de la categoría coincide con su nombre/código.
- **Selección de columnas finales e inserción:** se conservan únicamente los campos necesarios para la tabla `races` y se insertan con `to_sql()` en modo `append` (MySQL genera `id_race` automáticamente con `AUTO_INCREMENT`).
- **Creación del mapa de carreras (`races_map`):** se lee la tabla `races` ya insertada para obtener `id_race` junto con `year`, `sequence` e `id_category`. Este mapa se utilizará después para enlazar correctamente los registros en la tabla `results`.

In [115]:
# 9.1) Extraer carreras únicas para la tabla 'races':
races = df[['race_name', 'date', 'year', 'sequence', 'category', 'circuit_name']].drop_duplicates().dropna()

# 9.2) Aplicar el mapeo de circuits_map usando merge: 
races = races.merge(circuits_map, on='circuit_name', how='left')

# 9.3) En el caso de category tenemos directamente su id pues es básicamente su nombre:
races = races.rename(columns={'category': 'id_category'})

# 9.4) Escoger las columnas necesarias e insertar en la base de datos:
races = races[['race_name', 'date', 'year', 'sequence', 'id_category', 'id_circuit']]
races.to_sql('races', con=engine, if_exists='append', index=False)

# 9.5) Creamos el mapa de carreras para la tabla final de results:
races_map = pd.read_sql("SELECT id_race, year, sequence, id_category FROM races", con=engine)

# Al no enviar la columna 'id_race', MySQL usará el AUTO_INCREMENT automáticamente.
print("Carreras migradas exitosamente.")

Carreras migradas exitosamente.


#### **10) Results:**

Migramos los resultados finales a la tabla `results`, resolviendo previamente todas las referencias a otras tablas (integridad referencial):

- **Extracción de resultados únicos:** se seleccionan las columnas relevantes del CSV (velocidad, puntos, dorsal, tiempo, posición, piloto, moto, equipo y claves de la carrera: `year`, `sequence`, `category`) y se eliminan duplicados y nulos.
- **Mapeo de `id_race`:** se hace un `merge` con `races_map` usando la clave lógica de carrera (`year`, `sequence`, `category`) para obtener el identificador real `id_race` generado en MySQL.
- **Mapeo de `id_rider`:** se enlaza con `riders_map` a partir de `rider_first_name` y `rider_last_name` para obtener `id_rider`.
- **Mapeo de `id_bike`:** se enlaza con `bikes_map` usando el nombre de la moto (`bike`) para obtener `id_bike`.
- **Mapeo de `id_team`:** se enlaza con `teams_map` mediante `team_name` para obtener `id_team`.
- **Selección de columnas finales e inserción:** se seleccionan únicamente los campos requeridos por la tabla `results` (atributos del resultado + FKs) y se insertan con `to_sql()` en modo `append`. El campo `id_result` se genera automáticamente por `AUTO_INCREMENT`.

In [116]:
# 10.1) Extraer resultados únicos para la tabla 'results':
results = df[['speed', 'points', 'rider_number', 'time', 'position', 'rider_first_name', 
              'rider_last_name', 'bike', 'team_name', 'year', 'sequence', 'category']].drop_duplicates().dropna()

# 10.2) Mapear el id_race: 
results = results.merge(races_map, left_on=['year', 'sequence', 'category'], 
                        right_on=['year', 'sequence', 'id_category'], how='left')

# 10.3) Mapear el id_rider: 
results = results.merge(riders_map, left_on=['rider_first_name', 'rider_last_name'], 
                        right_on=['first_name', 'last_name'])

# 10.4) Mapear el id_bike: 
results = results.merge(bikes_map, left_on='bike', right_on='bike_name')

# 10.5) Mapear el id_team: 
results = results.merge(teams_map, on='team_name')

# 10.6) Escoger las columnas necesarias e insertar en la base de datos:
results = results[['speed', 'points', 'rider_number', 'time', 'position', 
                   'id_rider', 'id_bike', 'id_team', 'id_race']]
results.to_sql('results', con=engine, if_exists='append', index=False)

# Al no enviar la columna 'id_result', MySQL usará el AUTO_INCREMENT automáticamente.
print("Resultados migrados exitosamente.")

Resultados migrados exitosamente.


---

### Comprobación de la migración:

Validamos que la carga de datos en la tabla `results` se haya realizado correctamente:

In [117]:
# Comprobación: 
query = "SELECT * FROM results;"
resultado = pd.read_sql(query, con=engine)
resultado

,id_result,id_race,id_rider,id_team,id_bike,rider_number,position,points,speed,time
0,1,1,1,1,1,54,16,0.0,154.4,+37.590
1,2,1,886,1,1,54,16,0.0,154.4,+37.590
2,3,1,1771,1,1,54,16,0.0,154.4,+37.590
3,4,1,2656,1,1,54,16,0.0,154.4,+37.590
4,5,1,3541,1,1,54,16,0.0,154.4,+37.590
...,...,...,...,...,...,...,...,...,...,...
143689,143690,890,1311,362,13,21,14,2.0,152.3,+1'19.346
143690,143691,890,2196,362,13,21,14,2.0,152.3,+1'19.346
143691,143692,890,3081,362,13,21,14,2.0,152.3,+1'19.346
143692,143693,890,3966,362,13,21,14,2.0,152.3,+1'19.346
